In [4]:
import numpy as np
import tensorflow as tf
import unicodedata
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

In [5]:
def normalize(text):
  return unicodedata.normalize('NFD', text)

In [6]:
data = [("hello","नमस्ते"),("how are you","आप कैसे हैं"),
        ("i am fine","मैं ठीक हूँ"),("thank you","धन्यवाद"),
        ("good night","शुभ रात्रि")]

In [7]:
inp_texts, tgt_texts = [], []
inp_chars, tgt_chars = set(), set()

In [8]:
for i, t in data:
  i = normalize(i)
  t = normalize(t)
  t = '\t' + t + '\n'

  inp_texts.append(i)
  tgt_texts.append(t)

  inp_chars.update(i)
  tgt_chars.update(t)

In [9]:
inp_chars = sorted(inp_chars)
tgt_chars = sorted(tgt_chars)

in_tok = {c:i for i,c in enumerate(inp_chars)}
tg_tok = {c:i for i,c in enumerate(tgt_chars)}
rev_tg = {i:c for c, i in tg_tok.items()}

max_in = max(len(i) for i in inp_texts)
max_tg = max(len(t) for t in tgt_texts)

In [10]:
enc_in = np.zeros((len(inp_texts), max_in, len(inp_chars)))
dec_in = np.zeros((len(inp_texts), max_tg, len(tgt_chars)))
dec_tar = np.zeros_like(dec_in)

for i, (inp, tgt) in enumerate(zip(inp_texts, tgt_texts)):
  for t,c in enumerate(inp):
    enc_in[i, t, in_tok[c]] = 1
  for t, c in enumerate(tgt):
    dec_in[i, t, tg_tok[c]] = 1
    if t > 0:
      dec_tar[i, t-1, tg_tok[c]] = 1

In [11]:
latent = 256
enc_inputs = Input(shape=(None, len(inp_chars)))
_, h, c = LSTM(latent, return_state=True)(enc_inputs)
enc_states = [h, c]

In [13]:
dec_inputs = Input(shape=(None, len(tgt_chars)))
dec_lstm = LSTM(latent, return_sequences=True, return_state=True)
dec_out, _, _ = dec_lstm(dec_inputs, initial_state=enc_states)

dense = Dense(len(tgt_chars), activation='softmax')
dec_out = dense(dec_out)

model = Model([enc_inputs, dec_inputs], dec_out)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
model.fit([enc_in, dec_in], dec_tar, epochs=300, verbose=0)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 18)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, None, 29)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    281,600 │ input_layer[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, None,     │    292,864 │ input_layer_2[0]… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 29)  │      7,453 │ lstm_2[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,745,753 (6.66 MB)

 Trainable params: 581,917 (2.22 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,163,836 (4.44 MB)

In [14]:
enc_model = Model(enc_inputs, enc_states)

h_in = Input(shape=(latent,))
c_in = Input(shape=(latent,))

dec_out2, h, c = dec_lstm(dec_inputs, initial_state=[h_in, c_in])
dec_out2 = dense(dec_out2)

dec_model = Model(
    [dec_inputs, h_in, c_in],
    [dec_out2, h, c]
)

In [18]:
def decode(seq):
  states = enc_model.predict(seq, verbose=0)

  target = np.zeros((1,1, len(tgt_chars)))
  target[0,0,tg_tok['\t']] = 1

  result = ""

  while True:
    pred, h, c = dec_model.predict([target] + states, verbose=0)
    idx = np.argmax(pred[0,-1])
    char = rev_tg[idx]

    if char == '\n' or len(result) > max_tg:
      break
    result += char

    target = np.zeros((1,1, len(tgt_chars)))
    target[0,0,idx] = 1
    states = [h,c]
  return result

In [19]:
for i in range(len(inp_texts)):
  print(inp_texts[i],"->", decode(enc_in[i:i+1]))

hello -> नमस्ते
how are you -> आप कैसे हैं
i am fine -> मैं ठीक हूँ
thank you -> धन्यवाद
good night -> शुभ रात्रि


In [ ]:
""